In [24]:
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

In [52]:
%load_ext sql

In [36]:
spark.stop()

In [37]:
spark = (SparkSession
    .builder
    .config("spark.jars", r"../misc/postgresql-42.7.9.jar")
    .appName("hw01")
    .getOrCreate()
)

In [39]:
trip_data = spark.read.parquet("../data/green_tripdata_2025-11.parquet")
zone_lookup = spark.read.csv("../data/taxi_zone_lookup.csv", header=True)

In [40]:
trip_data.show(5)

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|cbd_congestion_fee|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------------+
|       2| 2025-11-01 00:34:48|  2025-11-01 00:41:39|                 N|         1|          74|          

In [41]:
trip_data.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- lpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- trip_type: long (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [48]:
trip_data.createOrReplaceTempView("trip_data")

In [49]:
zone_lookup.createOrReplaceTempView("zone_lookup")

In [59]:
spark.sql("""
select count(*) as total_trips
from trip_data
where true
    and (lpep_pickup_datetime >= '2025-11-01' and lpep_pickup_datetime < '2025-12-01')
    and trip_distance <= 1
""").show()

+-----------+
|total_trips|
+-----------+
|       8007|
+-----------+



In [64]:
spark.sql("""
select lpep_pickup_datetime, trip_distance
from trip_data
where true
    and trip_distance <= 100
ORDER BY trip_distance DESC
LIMIT 1
""").show()

+--------------------+-------------+
|lpep_pickup_datetime|trip_distance|
+--------------------+-------------+
| 2025-11-14 15:36:27|        88.03|
+--------------------+-------------+



In [67]:
spark.sql("""
select pickup_zone, sum(total_amount) total_amount
from trip_data
where true
    and (lpep_pickup_datetime >= '2025-11-18' and lpep_pickup_datetime < '2025-11-19')
GROUP BY ALL
""").show()

{"ts": "2026-01-24 00:44:49.991", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `pickup_zone` cannot be resolved. Did you mean one of the following? [`trip_type`, `ehail_fee`, `mta_tax`, `tip_amount`, `VendorID`]. SQLSTATE: 42703", "context": {"errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o188.sql.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `pickup_zone` cannot be resolved. Did you mean one of the following? [`trip_type`, `ehail_fee`, `mta_tax`, `tip_amount`, `VendorID`]. SQLSTATE: 42703; line 2 pos 7;\n'Aggregate ['ALL], ['pickup_zone, sum(total_amount#358) AS total_amount#732]\n+- Filter (true AND ((lpep_pickup_datetime#343 >= cast(2025-11-18 as timestamp_ntz)) AND (lpep_pickup_datetime#343 < cast(2025-11-19 as 

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `pickup_zone` cannot be resolved. Did you mean one of the following? [`trip_type`, `ehail_fee`, `mta_tax`, `tip_amount`, `VendorID`]. SQLSTATE: 42703; line 2 pos 7;
'Aggregate ['ALL], ['pickup_zone, sum(total_amount#358) AS total_amount#732]
+- Filter (true AND ((lpep_pickup_datetime#343 >= cast(2025-11-18 as timestamp_ntz)) AND (lpep_pickup_datetime#343 < cast(2025-11-19 as timestamp_ntz))))
   +- SubqueryAlias trip_data
      +- View (`trip_data`, [VendorID#342, lpep_pickup_datetime#343, lpep_dropoff_datetime#344, store_and_fwd_flag#345, RatecodeID#346L, PULocationID#347, DOLocationID#348, passenger_count#349L, trip_distance#350, fare_amount#351, extra#352, mta_tax#353, tip_amount#354, tolls_amount#355, ehail_fee#356, improvement_surcharge#357, total_amount#358, payment_type#359L, trip_type#360L, congestion_surcharge#361, cbd_congestion_fee#362])
         +- Relation [VendorID#3